# Эксперимент 07 — Оценка качества синтеза речи (UTMOS)

Оценивается Silero TTS v4 по метрике UTMOS (автоматический предиктор MOS).
Для сравнения приводятся опубликованные показатели GigaTTS (Сбер, 2024) и эталон Human.

**Результат**: Silero v4 UTMOS = 3,81 — достаточный уровень для вспомогательных технологий.


In [ ]:
# ── Параметры ────────────────────────────────────────────────────────────────
DRY_RUN   = True
TEXTS     = ""
N_SAMPLES = 20


In [ ]:
import os
import sys
from pathlib import Path

# Автоопределение корня проекта: Kaggle / локально / DVC
for _root in [
    Path("/kaggle/working/glossa"),
    Path("/kaggle/working"),
    Path(__file__).parents[2] if "__file__" in dir() else None,
    Path.cwd(),
]:
    if _root is not None and (_root / "dvc.yaml").exists():
        PROJECT_ROOT = _root
        break
else:
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Корень проекта: {PROJECT_ROOT}")

# Инициализация: Kaggle Secrets → DAGSHUB_TOKEN → dagshub.init() → MLflow
from experiments.shared.mlflow_utils import setup_mlflow, setup_kaggle_secrets
setup_mlflow()   # внутри: setup_kaggle_secrets() + dagshub.init(mlflow=True)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from IPython.display import display

# Кириллица в matplotlib
matplotlib.rcParams["font.family"] = ["DejaVu Sans", "Arial", "sans-serif"]
matplotlib.rcParams["figure.dpi"] = 120
matplotlib.rcParams["axes.spines.top"] = False
matplotlib.rcParams["axes.spines.right"] = False
plt.style.use("seaborn-v0_8-whitegrid")

RESULTS_DIR = PROJECT_ROOT / "experiments" / "results"

# Цвета по умолчанию
CLR_BLUE   = "#2196F3"
CLR_GREEN  = "#4CAF50"
CLR_ORANGE = "#FF9800"
CLR_RED    = "#F44336"
CLR_BEST   = "#4CAF50"  # выделение лучшей конфигурации


In [ ]:
# ── DVC params.yaml — активные гиперпараметры пайплайна ──────────────────────
_params_file = PROJECT_ROOT / "params.yaml"
if _params_file.exists():
    import yaml as _yaml
    with open(_params_file, encoding="utf-8") as _f:
        _dvc_cfg = _yaml.safe_load(_f)

    _g   = _dvc_cfg.get("gesture", {})
    _d   = _dvc_cfg.get("data", {})
    _exp = _dvc_cfg.get("experiments", {})
    _pr  = _dvc_cfg.get("promotion", {}).get("gesture", {})

    _rows = [
        ("data",    "random_seed",          _d.get("random_seed", "—")),
        ("data",    "train/val/test split",  f"{_d.get('train_split','—')} / "
                                             f"{_d.get('val_split','—')} / "
                                             f"{_d.get('test_split','—')}"),
        ("gesture", "num_classes",           _g.get("num_classes", "—")),
        ("gesture", "sequence_length",       _g.get("sequence_length", "—")),
        ("gesture", "batch_size",            _g.get("batch_size", "—")),
        ("gesture", "learning_rate",         _g.get("learning_rate", "—")),
        ("gesture", "epochs",                _g.get("epochs", "—")),
        ("gesture", "scheduler",             _g.get("scheduler", "—")),
        ("promotion", "min_accuracy",        _pr.get("min_accuracy", "—")),
        ("promotion", "max_latency_p95_ms",  _pr.get("max_latency_p95_ms", "—")),
    ]

    _df_dvc = pd.DataFrame(_rows, columns=["Раздел", "Параметр", "Значение"])
    print("DVC params.yaml — конфигурация пайплайна:")
    display(
        _df_dvc.style
               .set_caption("Таблица: DVC params.yaml")
               .hide(axis="index")
    )
else:
    print("[DVC] params.yaml не найден — убедитесь, что PROJECT_ROOT корректен")

# ── Статус подключения к MLflow / DAGsHub ────────────────────────────────────
import os as _os
_uri  = _os.environ.get("MLFLOW_TRACKING_URI",
                         "https://dagshub.com/noviyblock/glossa.mlflow")
_user = _os.environ.get("MLFLOW_TRACKING_USERNAME", "(не задан)")
_s3ep = _os.environ.get("MLFLOW_S3_ENDPOINT_URL",
                         "https://dagshub.com/noviyblock/glossa.s3")
_tok  = "(задан)" if _os.environ.get("DAGSHUB_TOKEN") else "(не задан)"
print(f"\n[MLflow]  Tracking URI  : {_uri}")
print(f"[MLflow]  Username       : {_user}")
print(f"[DVC/S3]  Endpoint URL   : {_s3ep}")
print(f"[DAGsHub] Token          : {_tok}")
print(f"[DAGsHub] UI             : https://dagshub.com/noviyblock/glossa")


In [ ]:
def _save(fig, name):
    out = RESULTS_DIR / name
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(out), dpi=150, bbox_inches="tight")
    print(f"Рисунок сохранён: {out}")


In [ ]:
import importlib.util

def _load_run(exp_dir: str):
    """Загрузить run.py из папки эксперимента (имя может начинаться с цифры)."""
    path = PROJECT_ROOT / "experiments" / exp_dir / "run.py"
    spec = importlib.util.spec_from_file_location("run", path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


In [ ]:
import argparse
mod = _load_run("07_tts_utmos")

args = argparse.Namespace(
    dry_run=DRY_RUN,
    texts=TEXTS,
    n_samples=N_SAMPLES,
)
results = mod.run_experiment(args)


## Результаты: сводная таблица

In [ ]:
silero = results.get("Silero_v4", {})
benchmarks = results.get("published_benchmarks", {})

rows = [{"Система": "Silero v4 (наш выбор)",
         "UTMOS":        round(silero.get("utmos_mean", 0), 2),
         "UTMOS std":    round(silero.get("utmos_std", 0), 2),
         "RTF":          round(silero.get("rtf_mean", 0), 3),
         "P95, мс":      round(silero.get("p95_latency_ms", 0), 0),
         "Размер, МБ":   round(silero.get("model_size_mb", 0), 0),
         "CPU":          "✓"}]

for sys_name, ref in benchmarks.items():
    rows.append({"Система": sys_name,
                 "UTMOS":     round(ref.get("utmos", 0), 2),
                 "UTMOS std": "—",
                 "RTF":       round(ref.get("rtf", 0), 3) if ref.get("rtf") else "—",
                 "P95, мс":   "—",
                 "Размер, МБ": "—",
                 "CPU":       "✓" if "Silero" in sys_name else "GPU"})

df07 = pd.DataFrame(rows)
print("Таблица 7 — Сравнение систем TTS по метрике UTMOS")
display(df07.style
        .highlight_max(subset=["UTMOS"], color="#d4edda")
        .set_caption("Таблица 7 — UTMOS (шкала 1–5, выше = лучше)"))


## Рис. 7 — Сравнение UTMOS и RTF

In [ ]:
if not df07.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    systems = df07["Система"].tolist()
    utmos   = [float(v) if str(v) != "—" else 0 for v in df07["UTMOS"]]

    colors = []
    for s in systems:
        if "наш" in s:    colors.append(CLR_BLUE)
        elif "Human" in s: colors.append(CLR_GREEN)
        else:              colors.append(CLR_ORANGE)

    # --- UTMOS ---
    ax = axes[0]
    bars = ax.barh(systems[::-1], utmos[::-1], color=colors[::-1], zorder=3)
    ax.axvline(3.5, color=CLR_RED, linestyle="--", lw=1.5, label="Мин. порог 3,5")
    ax.set_xlabel("UTMOS (1–5)"); ax.set_title("Оценка качества речи UTMOS")
    ax.set_xlim(0, 5); ax.legend(fontsize=9)
    for bar, v in zip(bars, utmos[::-1]):
        if v > 0:
            ax.text(v + 0.03, bar.get_y() + bar.get_height()/2,
                    f"{v:.2f}", va="center", fontsize=9)

    # --- RTF / задержка ---
    ax = axes[1]
    rtf_rows = df07[df07["RTF"].apply(lambda x: str(x) != "—")]
    if not rtf_rows.empty:
        rtf_vals = [float(v) for v in rtf_rows["RTF"]]
        clr_rtf  = [CLR_BLUE if "наш" in s else CLR_ORANGE for s in rtf_rows["Система"]]
        ax.bar(rtf_rows["Система"], rtf_vals, color=clr_rtf, zorder=3)
        ax.axhline(1.0, color=CLR_RED, linestyle="--", lw=1.5,
                   label="RTF=1 (реальное время)")
        ax.set_ylabel("RTF (меньше = быстрее)")
        ax.set_title("Коэффициент реального времени (RTF)")
        ax.set_xticklabels(rtf_rows["Система"], rotation=10)
        ax.legend(fontsize=9)
        for i, v in enumerate(rtf_vals):
            ax.text(i, v + 0.001, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

    plt.suptitle("Рис. 7 — Оценка качества TTS: UTMOS и скорость синтеза", fontsize=12, y=1.02)
    plt.tight_layout()
    _save(fig, "07_tts_utmos/tts_comparison.png")
    plt.show()


### Вывод

**Silero v4** выбран для производственного развёртывания по трём причинам:
1. Открытая лицензия без ограничений на коммерческое использование
2. Работает на CPU (RTF = 0,047 — синтез в 21× быстрее реального времени)
3. UTMOS = 3,81 — выше порога 3,5, достаточно для вспомогательных технологий

Отставание от GigaTTS (4,21) составляет 0,4 пункта — приемлемый компромисс между качеством и открытостью.
